In [ ]:
import numpy as np
import random
import socket

# Optional deterministic behavior
SEED = 42
if SEED is not None:
    np.random.seed(SEED)
    random.seed(SEED)


In [ ]:
class Perceptron:
    def __init__(self, kind, action=None):
        self.kind = kind
        self.action = action
        self.utility = 1.0
        self.weights = None
        self.eligibility = 0.0  # temporal trace
        self.decay_rate = 0.0  # new asymmetric decay parameter

    def ensure_weights(self, dim):
        if self.weights is None:
            self.weights = np.random.randn(dim) * 0.01

    def predict(self, state):
        self.ensure_weights(len(state))
        return np.dot(self.weights, state)

    def update(self, state, error, gamma=0.9, action_type=None, stagnation=0.0):
        self.ensure_weights(len(state))
        self.eligibility = gamma * self.eligibility + 1.0

        lr = 0.03
        self.weights += lr * error * state * self.eligibility

        # --- asymmetric decay ---
        if action_type == "wait":
            self.utility *= (1.0 - 0.01 * stagnation)
        elif action_type == "move":
            self.utility *= (1.0 - 0.002 * stagnation)

        # keep utility bounded
        self.utility = np.clip(self.utility, 0.01, 2.0)



In [ ]:
# --- Brain + anticipatory action with stochastic exploration and asymmetric decay ---
class Brain:
    def __init__(self):
        self.perceptrons = []
        self.prev_signals = []

    def add(self, p):
        self.perceptrons.append(p)

    def actions(self):
        return [p for p in self.perceptrons if p.kind == "action"]

    def objectives(self):
        return [p for p in self.perceptrons if p.kind == "objective"]

    def entities(self):
        return [p for p in self.perceptrons if p.kind == "entity"]

    def stagnation_level(self, window=5):
        """Measure how stagnant the signal has been recently (0 = no stagnation)."""
        if len(self.prev_signals) < 2:
            return 0.0
        recent = self.prev_signals[-window:]
        diffs = [abs(j - i) for i, j in zip(recent[:-1], recent[1:])]
        return 1.0 - (sum(diffs) / (len(diffs) + 1e-6))  # high if little change

    def predict_future_error(self, state, action):
        """
        Predict future signal change for a given action.
        Adds stochastic boost to 'move' to reduce procrastination.
        """
        delta = 0.0
        for e in self.entities():
            delta += e.predict(state)

        # Action heuristics
        if action.action == "move":
            delta *= 0.5
            # stochastic boost for exploration
            stag = self.stagnation_level()
            delta += np.random.rand() * 0.2 * stag + np.random.rand() * 0.05
        elif action.action == "wait":
            delta *= 0.1
            delta += np.random.randn() * 0.01  # tiny noise

        return delta

    def learn(self, state, next_state, dead=False, gamma=0.9):
        error_vector = next_state - state
        error = np.linalg.norm(error_vector)
        if dead:
            error *= 0.5

        for p in self.perceptrons:
            p.update(state, -error, gamma=gamma)

        # novelty detection
        best_novelty = min((abs(e.predict(state)) for e in self.entities()), default=float("inf"))
        if best_novelty > 0.4:
            for e in self.entities():
                sim = np.linalg.norm(e.weights - np.random.randn(len(state)) * 0.01)
                if sim < 0.3:
                    break
            else:
                self.add(Perceptron("entity"))

        # prune low-utility perceptrons
        self.perceptrons = [p for p in self.perceptrons if p.utility > 0.05]

    def log_signal(self, signal):
        """Keep track of previous raw signals for stagnation computation."""
        self.prev_signals.append(signal)
        if len(self.prev_signals) > 50:  # memory cap
            self.prev_signals.pop(0)


def anticipatory_action(brain, state, exploration_weight=1.5):
    """
    Choose action based on predicted future error with exploration and stochastic boost.
    """
    best_action, best_score = None, -np.inf
    for a in brain.actions():
        predicted = brain.predict_future_error(state, a)

        # exploration bias
        if a.action == "move":
            predicted *= exploration_weight

        if predicted > best_score:
            best_score = predicted
            best_action = a

    # fallback if no action exceeds threshold
    return best_action if best_action else np.random.choice(brain.actions())


In [ ]:
class ToyWorld:
    def __init__(self):
        # raw internal signal, unknown to agent
        self.internal_signal = random.uniform(0.3, 0.7)
        # hidden environment mode (agent never sees this)
        self.hidden_mode = random.choice([0, 1])

    def state(self):
        # agent only sees raw numbers; no semantic label
        return np.array([self.internal_signal, float(self.hidden_mode)])

    def step(self, action):
        # environment dynamics
        if action == "move":
            if self.hidden_mode == 0:
                self.internal_signal -= 0.15
            else:
                self.internal_signal += 0.15
        elif action == "wait":
            # small natural drift
            self.internal_signal += random.uniform(-0.02, 0.02)

        # stochastic events
        if random.random() < 0.1:  # 10% chance of random jump
            self.internal_signal += random.uniform(-0.2, 0.2)
        if random.random() < 0.05:  # 5% chance of random mode flip
            self.hidden_mode = 1 - self.hidden_mode

        # minor Gaussian noise
        self.internal_signal += np.random.randn() * 0.01

        # clamp signal
        self.internal_signal = np.clip(self.internal_signal, 0.0, 1.0)

        # survival penalty if too low
        dead = self.internal_signal <= 0.05
        return dead


In [ ]:
def temporal_state(prev_states, current_state, window=5, alpha=0.7):
    """
    Build a weighted temporal state vector emphasizing recent states.
    Older states get decayed by alpha**i.
    """
    # copy history
    history = prev_states[-(window-1):] if len(prev_states) > 0 else []

    # pad with zeros if needed
    while len(history) < window - 1:
        history.insert(0, np.zeros_like(current_state))

    # append current state
    history = history + [current_state]

    # apply exponential decay weighting
    weighted_states = [s * (alpha**(window - 1 - i)) for i, s in enumerate(history)]
    return np.concatenate(weighted_states)


In [ ]:
def anticipatory_action(brain, state, exploration_weight=1.5, forced_explore_prob=0.3, explore_bonus=0.3):
    """
    Select action based on predicted future error with exploration bias.
    - exploration_weight favors 'move'
    - forced_explore_prob ensures random exploration sometimes
    - explore_bonus adds flat boost to move actions
    """
    # Forced random exploration
    if random.random() < forced_explore_prob:
        return np.random.choice(brain.actions())

    best_action, best_score = None, -np.inf

    for a in brain.actions():
        predicted = brain.predict_future_error(state, a)

        # bias exploration actions
        if a.action == "move":
            predicted *= exploration_weight
            predicted += explore_bonus  # flat boost to favor move

        # cap wait so it doesn't dominate when stagnation is low
        if a.action == "wait":
            predicted = min(predicted, 0.6)

        if predicted > best_score:
            best_score = predicted
            best_action = a

    return best_action if best_action else np.random.choice(brain.actions())


In [ ]:
# --- Episode loop with forced exploration, exploration bonus, and stochastic environment ---
num_episodes = 5
temporal_window = 3
alpha_decay = 0.7
exploration_weight = 1.5
forced_explore_prob = 0.5  # chance of forced random exploration
explore_bonus = 0.3        # bonus added to 'move' score

In [ ]:
brain = Brain()
brain.add(Perceptron("objective"))  # exploration
brain.add(Perceptron("objective"))  # survival
brain.add(Perceptron("action", action="move"))
brain.add(Perceptron("action", action="wait"))


state_dim = 3
temporal_length = 3
prev_states = [np.zeros(state_dim) for _ in range(temporal_length)]


In [ ]:
HOST = "127.0.0.1"
PORT = 5005
server = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
server.bind((HOST, PORT))
server.listen(1)
print(f"Waiting for Lua script connection on {HOST}:{PORT}...")
conn, addr = server.accept()
print(f"Connected to Lua: {addr}")


In [ ]:
import socket
import numpy as np
import random

HOST = "127.0.0.1"
PORT = 5005
server = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
server.bind((HOST, PORT))
server.listen(1)
print(f"Waiting for Lua connection on {HOST}:{PORT}...")
conn, addr = server.accept()
print(f"Connected to Lua: {addr}")

# --- Define available actions ---
gba_buttons = ["A", "B", "Up", "Down", "Left", "Right", "Start", "Select"]

# --- Define Brain, Perceptron etc. as before ---
# (Keep your Perceptron, Brain, anticipatory_action definitions)

# Example: state_dim matches Lua stream length
state_dim = 4  # player HP, player X/Y, enemy HP
temporal_window = 3
prev_states = [np.zeros(state_dim) for _ in range(temporal_window)]

# Episode loop
num_episodes = 5
for ep in range(num_episodes):
    print(f"\n=== Episode {ep+1} ===")
    t = 0
    while True:
        try:
            data = conn.recv(1024).decode()
            if not data:
                continue
            lines = data.strip().split("\n")
            for line in lines:
                values = np.array([float(x) for x in line.split(",")])
                prev_states.pop(0)
                prev_states.append(values)
                state = temporal_state(prev_states, values, window=temporal_window, alpha=0.7)

                # choose action (returns a Perceptron with action.name matching a GBA button)
                action = anticipatory_action(brain, state, exploration_weight=1.5)

                # Send button command to Lua
                button_cmd = action.action if action else "None"
                conn.send(f"{button_cmd}\n".encode())

                # Learn
                brain.learn(state, state.copy())  # next_state same for now
                brain.log_signal(values[0])  # track player HP or primary signal

                print(f"t={t:03d} | button={button_cmd} | state={values}")
                t += 1
                if t > 200:  # optional cutoff
                    break
        except Exception as e:
            print("Error:", e)
            continue


In [ ]:
conn.close()
server.close()
print("Socket closed.")
